# 🏪 Digital Sari-Sari Store — Reporting & Visualization

A live analytics notebook over the **3NF PostgreSQL warehouse**. It queries the
database directly (through the shared `sarisari.reporting` layer — the *same*
query code the Streamlit dashboard uses) and renders the three reports asked of
the project:

1. **Sales** — revenue / units / baskets over time, with **time-range, product-group
   and brand** filters.
2. **Customers** — per-customer **utang (credit)** balances, payments, transactions
   and amounts.
3. **Stock** — on-hand levels, reorder flags and inventory value.

> Every derived number (balances, stock) is read from the database **views**
> (`v_customer_balances`, `v_utang_ledger`, `v_product_stock`) — the notebook
> never re-implements the business math.

**Prerequisite:** the pipeline has been run (`make pipeline`) so the warehouse is
loaded. Launch with `make notebook` (or `jupyter lab`).

## 0 · Setup & connection

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

from sarisari import reporting as R

pio.renderers.default = "notebook"          # embed Plotly so charts render offline
pd.options.display.float_format = lambda v: f"{v:,.2f}"

PESO = "\u20b1"
def peso(x):    return f"{PESO}{x:,.2f}"
def peso_k(x):
    if abs(x) >= 1e6:  return f"{PESO}{x/1e6:,.2f}M"
    if abs(x) >= 1e3:  return f"{PESO}{x/1e3:,.1f}k"
    return f"{PESO}{x:,.0f}"

engine = R.get_engine()
dim    = R.load_dim_products(engine)        # 187-SKU product dimension (+ derived brand)
lo, hi = R.date_bounds(engine)
print(f"Connected. Data window: {lo} -> {hi}")
print(f"{len(dim)} products across {dim['category'].nunique()} groups "
      f"and {dim['brand'].nunique()} brands.")

## 1 · Filters  ⟵ *edit these and re-run the notebook*

The whole notebook is driven by a single `R.Filters` object: a **date window**,
an optional list of **product groups** (categories) and an optional list of
**brands**. Empty list = no restriction.

In [ ]:
# --- Tweak these ------------------------------------------------------------
START      = lo                 # e.g. date(2020, 1, 1)
END        = hi                 # e.g. date(2026, 6, 28)
CATEGORIES = []                 # e.g. ["Liquor", "Beverages"]   (empty = all groups)
BRANDS     = []                 # e.g. ["Tanduay", "San Miguel"] (empty = all brands)
FREQ       = "Monthly"          # Daily | Weekly | Monthly | Quarterly | Yearly
# ----------------------------------------------------------------------------

flt = R.Filters(start=START, end=END,
                categories=tuple(CATEGORIES), brands=tuple(BRANDS))
print("Active slice:", flt)
print("\nAvailable groups:", R.list_categories(engine))
print("\nTop 15 brands by SKU count:", R.list_brands(dim)[:15])

## 2 · Sales analysis\n\nHeadline KPIs for the selected slice.

In [ ]:
k = R.kpi_summary(engine, flt, dim)
pd.DataFrame({
    "Metric": ["Revenue", "Units sold", "Transactions", "Avg. basket",
               "Credit revenue", "Credit share", "Active customers"],
    "Value":  [peso_k(k['revenue']), f"{k['units']:,}", f"{k['transactions']:,}",
               peso(k['avg_basket']), peso_k(k['credit_revenue']),
               f"{k['credit_share']*100:.1f}%", f"{k['active_customers']:,}"],
}).set_index("Metric")

### 2.1 · Trend over time (time filter)

In [ ]:
ts = R.sales_timeseries(engine, flt, dim, FREQ)
fig = px.area(ts, x="period", y="revenue", markers=(len(ts) <= 60),
              title=f"Revenue per {FREQ.lower()} — {START} to {END}")
fig.update_traces(line_color="#2a9d8f", fillcolor="rgba(42,157,143,0.15)")
fig.update_layout(height=380, xaxis_title=None, yaxis_title="Revenue (\u20b1)")
fig.show()

In [ ]:
# Units & transactions on the same timeline
fig = px.line(ts, x="period", y=["units", "transactions"], markers=(len(ts) <= 60),
              title=f"Units & transactions per {FREQ.lower()}")
fig.update_layout(height=340, xaxis_title=None, yaxis_title="Count",
                  legend_title_text="")
fig.show()

### 2.2 · By product group (group filter)

In [ ]:
cat = R.sales_by_category(engine, flt, dim)
fig = px.bar(cat.sort_values("revenue"), x="revenue", y="category", orientation="h",
             color="category", color_discrete_sequence=px.colors.qualitative.Set2,
             text_auto=".2s", title="Revenue by product group")
fig.update_layout(showlegend=False, height=320, xaxis_title="Revenue (\u20b1)",
                  yaxis_title=None)
fig.show()
cat

### 2.3 · By brand (brand filter)

In [ ]:
brand = R.sales_by_brand(engine, flt, dim).head(15)
fig = px.bar(brand.sort_values("revenue"), x="revenue", y="brand", orientation="h",
             color="revenue", color_continuous_scale="Teal", text_auto=".2s",
             title="Top 15 brands by revenue")
fig.update_layout(height=460, coloraxis_showscale=False,
                  xaxis_title="Revenue (\u20b1)", yaxis_title=None)
fig.show()

### 2.4 · Cash vs. credit & best-selling SKUs

In [ ]:
mix = R.payment_mix(engine, flt, dim)
fig = px.pie(mix, names="payment_type", values="revenue", hole=0.55,
             color="payment_type",
             color_discrete_map={"cash": "#264653", "credit": "#e76f51"},
             title="Revenue: cash vs. credit (utang)")
fig.update_layout(height=340)
fig.show()

In [ ]:
R.top_products(engine, flt, dim, n=15)

## 3 · Customer analysis (utang / credit)

Per-customer **credit, transactions and amounts**, read from
`v_customer_balances` (running balance) joined with lifetime activity.

In [ ]:
bal = R.customer_balances(engine)
outstanding   = bal['balance'].clip(lower=0).sum()
owing         = int((bal['balance'] > 0.005).sum())
over_limit    = int((bal['credit_available'] < -0.005).sum())
print(f"Outstanding utang across the store : {peso(outstanding)}")
print(f"Customers currently owing          : {owing:,}")
print(f"Customers over their credit limit  : {over_limit:,}")
bal.head(10)

### 3.1 · Top debtors & balance distribution

In [ ]:
top = bal[bal['balance'] > 0].head(15).copy()
top['who'] = top['nickname'].fillna(top['full_name'])
fig = px.bar(top.sort_values("balance"), x="balance", y="who", orientation="h",
             color="balance", color_continuous_scale="Reds", text_auto=".2s",
             title="Top 15 debtors (running utang)")
fig.update_layout(height=420, coloraxis_showscale=False,
                  xaxis_title="Balance (\u20b1)", yaxis_title=None)
fig.show()

In [ ]:
fig = px.histogram(bal[bal['balance'] > 0], x="balance", nbins=40,
                   title="Distribution of outstanding balances")
fig.update_traces(marker_color="#e76f51")
fig.update_layout(height=340, xaxis_title="Balance (\u20b1)", yaxis_title="Customers")
fig.show()

### 3.2 · Single-customer drill-down (the digital *utang* page)

In [ ]:
# Pick the customer with the largest balance (change CUSTOMER_ID to explore)
CUSTOMER_ID = int(bal.iloc[0]['customer_id'])

d = R.customer_detail(engine, CUSTOMER_ID)
print(f"Customer #{CUSTOMER_ID}: {d['full_name']}"
      + (f" ('{d['nickname']}')" if d.get('nickname') else ""))
print(f"  Balance (utang) : {peso(float(d['balance']))}")
print(f"  Credit limit    : {peso(float(d['credit_limit']))}"
      f"   (headroom {peso(float(d['credit_available']))})")
print(f"  Lifetime spend  : {peso_k(float(d['lifetime_spend']))}"
      f"   over {int(d['n_transactions']):,} transactions")
print(f"  Active          : {d.get('first_purchase')} -> {d.get('last_purchase')}")

In [ ]:
ledger = R.customer_ledger(engine, CUSTOMER_ID)
fig = px.line(ledger, x="entry_date", y="running_balance", markers=True,
              title=f"Running utang balance — customer #{CUSTOMER_ID}")
fig.update_traces(line_color="#e76f51")
fig.update_layout(height=340, xaxis_title=None, yaxis_title="Balance (\u20b1)")
fig.show()
ledger.tail(10)

In [ ]:
# Most recent transactions (amounts & item counts) for this customer
R.customer_transactions(engine, CUSTOMER_ID, limit=15)

## 4 · Stock analysis

On-hand levels from `v_product_stock` (strictly-derived, point-in-time) plus
inventory value (on-hand × current price). The **group / brand** filters apply;
the date window does not (stock is cumulative).

In [ ]:
stock = R.stock_report(engine, flt, dim)
print(f"SKUs in view        : {len(stock):,}")
print(f"Units on hand       : {int(stock['on_hand'].sum()):,}")
print(f"Inventory value     : {peso_k(stock['stock_value'].sum())}")
print(f"Products to reorder : {int(stock['needs_reorder'].sum()):,}")
stock.head(10)

### 4.1 · Inventory value by group & lowest-stock items

In [ ]:
by_cat = stock.groupby("category", as_index=False)["stock_value"].sum()
fig = px.bar(by_cat.sort_values("stock_value"), x="stock_value", y="category",
             orientation="h", color="category",
             color_discrete_sequence=px.colors.qualitative.Set2, text_auto=".2s",
             title="Inventory value by product group")
fig.update_layout(showlegend=False, height=320, xaxis_title="Value (\u20b1)",
                  yaxis_title=None)
fig.show()

In [ ]:
low = stock.nsmallest(15, "on_hand")
fig = px.bar(low.sort_values("on_hand", ascending=False),
             x="on_hand", y="product", orientation="h", text_auto=True,
             color="on_hand", color_continuous_scale="OrRd_r",
             title="15 lowest-stock products")
fig.update_layout(height=420, coloraxis_showscale=False, xaxis_title="On hand",
                  yaxis_title=None)
fig.show()

### 4.2 · Reconciliation — derived vs. trigger-maintained stock\n\nThe schema keeps stock *both* ways on purpose; here we confirm they agree.

In [ ]:
drift = stock.assign(drift=stock['on_hand'] - stock['maintained_on_hand'])
mismatches = int((drift['drift'] != 0).sum())
print(f"Products where derived on_hand != maintained_on_hand: {mismatches}")
cols = ['product', 'on_hand', 'maintained_on_hand', 'drift']
drift[drift['drift'] != 0][cols] if mismatches else "All products reconcile exactly."

---

*The same `sarisari.reporting` functions power the interactive web dashboard —
run it with* `make dashboard` *(or* `streamlit run dashboard/app.py`*).*